# Tutorial: Backfill Missing Games with NBA PlayByPlayV3

Audience:
- GLA maintainers who need to backfill missing raw play-by-play rows for specific NBA games.

Prerequisites:
- `nba_api` and `pandas` installed in this notebook environment.
- Write access to `/Users/robschoen/Dropbox/CC/NBA_Data` if you want to merge into season-level files.

Learning goals:
- Fetch one game from `nba_api.stats.endpoints.playbyplayv3`.
- Normalize rows to the GLA canonical `api_pbpv3` schema.
- Save a one-off extract and optionally merge it into season CSVs.


## Outline

1. Environment setup and imports.
2. Configure the game ID and output behavior.
3. Fetch PlayByPlayV3 with retries.
4. Normalize and validate the data.
5. Save one-off files and optionally append to season CSV.


In [127]:
from __future__ import annotations

from pathlib import Path
import json
import random
import time

import pandas as pd
import requests

try:
    from nba_api.stats.endpoints import playbyplayv3
    from nba_api.stats.library.http import STATS_HEADERS as NBA_STATS_HEADERS
except Exception as exc:
    raise RuntimeError(
        "Could not import nba_api PlayByPlayV3. Install with: pip install nba_api pandas"
    ) from exc

# live endpoint imports are optional; we can still run without them.
try:
    from nba_api.live.nba.endpoints import playbyplay as live_playbyplay
    from nba_api.live.nba.library.http import STATS_HEADERS as NBA_LIVE_HEADERS
except Exception:
    live_playbyplay = None
    NBA_LIVE_HEADERS = {
        "User-Agent": "Mozilla/5.0",
        "Accept": "application/json",
    }

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)

"Imports loaded."


'Imports loaded.'

## Step 1 - Configure one-game backfill inputs

Set `GAME_ID` to any 10-digit NBA game ID.

- `APPEND_TO_SEASON_FILE = False` keeps this notebook in safe read/write mode for one-off exports only.
- Flip it to `True` when you are ready to merge the game into your `NBA_Data/PBPdata/api_pbpv3/...` season file.


In [128]:
GAME_ID = "0021201230"  # Example: regular season game id  # 10-digit NBA game id
SEASON = "2012-13"      # Used only for season-file target path      # Used only for season-file target path
PHASE = "regular"       # "regular" or "playoffs"       # "regular" or "playoffs"

REQUEST_TIMEOUT = 30
RETRIES = 3
BACKOFF_BASE_SECONDS = 0.75
USE_CDN_FALLBACK = True

# Set only if your network requires an HTTP(S) proxy.
# Example: PROXY = "http://127.0.0.1:8888"
PROXY = None

SAVE_ONE_OFF = True
ONE_OFF_DIR = Path("/Users/robschoen/Dropbox/CC/GLA/data/pbp/raw_backfill")

APPEND_TO_SEASON_FILE = False
NBA_DATA_REPO = Path("/Users/robschoen/Dropbox/CC/NBA_Data")

if PHASE not in {"regular", "playoffs"}:
    raise ValueError("PHASE must be 'regular' or 'playoffs'.")

GAME_ID


'0021201230'

## Step 2 - Define canonical schema helpers

These helpers mirror your project's canonical `api_pbpv3` normalization (column order, dtypes, sorting, and game ID formatting).


In [129]:
PBPV3_CANONICAL_COLUMNS = [
    "actionNumber",
    "clock",
    "period",
    "teamId",
    "teamTricode",
    "personId",
    "playerName",
    "playerNameI",
    "xLegacy",
    "yLegacy",
    "shotDistance",
    "shotResult",
    "isFieldGoal",
    "scoreHome",
    "scoreAway",
    "pointsTotal",
    "location",
    "description",
    "actionType",
    "subType",
    "videoAvailable",
    "shotValue",
    "actionId",
    "gameId",
]


def normalize_game_id(game_id: object) -> str:
    if pd.isna(game_id):
        return ""
    gid = str(game_id).strip()
    if gid.endswith(".0"):
        gid = gid[:-2]
    digits = "".join(ch for ch in gid if ch.isdigit())
    if not digits:
        return gid
    return digits.zfill(10)


def normalize_api_pbpv3_df(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()

    for col in PBPV3_CANONICAL_COLUMNS:
        if col not in d.columns:
            d[col] = pd.NA

    d = d[PBPV3_CANONICAL_COLUMNS]
    d["gameId"] = d["gameId"].map(normalize_game_id)
    d = d[d["gameId"] != ""].copy()

    int_default_zero_cols = [
        "actionNumber",
        "period",
        "teamId",
        "personId",
        "xLegacy",
        "yLegacy",
        "shotDistance",
        "isFieldGoal",
        "pointsTotal",
        "videoAvailable",
        "shotValue",
    ]
    for col in int_default_zero_cols:
        d[col] = pd.to_numeric(d[col], errors="coerce").fillna(0).astype("int64")

    d["scoreHome"] = pd.to_numeric(d["scoreHome"], errors="coerce")
    d["scoreAway"] = pd.to_numeric(d["scoreAway"], errors="coerce")

    action_id_numeric = pd.to_numeric(d["actionId"], errors="coerce")
    d["actionId"] = action_id_numeric.fillna(d["actionNumber"]).astype("int64")

    d = d.sort_values(["gameId", "actionNumber"], kind="stable").reset_index(drop=True)
    return d


def season_target_path(season: str, phase: str, repo_dir: Path) -> Path:
    start_year = int(season.split("-")[0])
    if phase == "regular":
        filename = f"api_pbpv3_{start_year}.csv"
    else:
        filename = f"api_pbpv3_po_{start_year}.csv"
    return repo_dir / "PBPdata" / "api_pbpv3" / phase / filename


print(f"Canonical columns: {len(PBPV3_CANONICAL_COLUMNS)}")


Canonical columns: 24


## Step 3 - Fetch one game from PlayByPlayV3


In [130]:
def _short_error(exc: Exception, max_len: int = 220) -> str:
    msg = f"{type(exc).__name__}: {exc}".replace("\n", " ").strip()
    return msg[:max_len] + ("..." if len(msg) > max_len else "")


def _proxy_dict(proxy: str | None) -> dict[str, str] | None:
    if proxy and str(proxy).strip():
        return {"http": str(proxy), "https": str(proxy)}
    return None


def _extract_actions(payload: dict, source_name: str) -> pd.DataFrame:
    actions = payload.get("game", {}).get("actions", [])
    if not isinstance(actions, list) or not actions:
        raise RuntimeError(f"{source_name} returned no actions")

    out = pd.DataFrame(actions)
    if out.empty:
        raise RuntimeError(f"{source_name} returned an empty dataframe")
    return out


def fetch_from_live_endpoint(game_id: str, timeout: float, proxy: str | None = None) -> pd.DataFrame:
    if live_playbyplay is None:
        raise RuntimeError("nba_api live endpoint module is unavailable in this environment")

    headers = dict(NBA_LIVE_HEADERS)
    headers.setdefault("Referer", "https://www.nba.com/")
    headers.setdefault("Origin", "https://www.nba.com")

    live_resp = live_playbyplay.PlayByPlay(
        game_id=game_id,
        timeout=timeout,
        proxy=proxy,
        headers=headers,
    )
    payload = live_resp.get_dict()
    out = _extract_actions(payload, "nba_api live")
    if "gameId" not in out.columns:
        out["gameId"] = game_id
    return out


def fetch_from_cdnnba_requests(game_id: str, timeout: float, proxy: str | None = None) -> pd.DataFrame:
    url = f"https://cdn.nba.com/static/json/liveData/playbyplay/playbyplay_{game_id}.json"
    headers = dict(NBA_LIVE_HEADERS)
    headers.setdefault("Referer", "https://www.nba.com/")
    headers.setdefault("Origin", "https://www.nba.com")
    headers.pop("Accept-Encoding", None)

    resp = requests.get(
        url,
        headers=headers,
        timeout=timeout,
        proxies=_proxy_dict(proxy),
    )
    if resp.status_code == 403:
        raise RuntimeError("HTTP 403 from cdn.nba.com (network/firewall block likely)")
    resp.raise_for_status()

    payload = resp.json()
    out = _extract_actions(payload, "cdnnba")
    if "gameId" not in out.columns:
        out["gameId"] = game_id
    return out


def fetch_single_game_pbp(
    game_id: str,
    timeout: float,
    retries: int,
    backoff_base_seconds: float,
    use_cdn_fallback: bool = True,
    proxy: str | None = None,
) -> tuple[pd.DataFrame, str]:
    game_id = normalize_game_id(game_id)
    if len(game_id) != 10:
        raise ValueError(f"Expected a 10-digit game id, got: {game_id!r}")

    attempt_count = max(1, int(retries))
    errors: list[str] = []

    for attempt in range(1, attempt_count + 1):
        try:
            stats_headers = dict(NBA_STATS_HEADERS)
            stats_headers.setdefault("Referer", "https://stats.nba.com/")
            response = playbyplayv3.PlayByPlayV3(
                game_id=game_id,
                timeout=timeout,
                proxy=proxy,
                headers=stats_headers,
            )
            dfs = response.get_data_frames()
            if dfs and dfs[0] is not None and not dfs[0].empty:
                out = dfs[0].copy()
                if "gameId" not in out.columns:
                    out["gameId"] = game_id
                return out, "nba_api_stats"
            errors.append(f"nba_api stats attempt {attempt}: empty response")
        except Exception as exc:
            errors.append(f"nba_api stats attempt {attempt}: {_short_error(exc)}")

        if use_cdn_fallback:
            try:
                out = fetch_from_live_endpoint(game_id, timeout=timeout, proxy=proxy)
                return out, "nba_api_live"
            except Exception as exc:
                errors.append(f"nba_api live attempt {attempt}: {_short_error(exc)}")

            try:
                out = fetch_from_cdnnba_requests(game_id, timeout=timeout, proxy=proxy)
                return out, "cdnnba_requests"
            except Exception as exc:
                errors.append(f"cdnnba direct attempt {attempt}: {_short_error(exc)}")

        if attempt < attempt_count:
            sleep_seconds = backoff_base_seconds * (2 ** (attempt - 1)) + random.uniform(0.0, 0.35)
            time.sleep(sleep_seconds)

    hint = ""
    if not proxy:
        hint = " | hint: if your network requires a proxy, set PROXY (for example http://127.0.0.1:8888)."
    raise RuntimeError("PlayByPlayV3 fetch failed. " + " | ".join(errors[-9:]) + hint)


raw_df, source_used = fetch_single_game_pbp(
    game_id=GAME_ID,
    timeout=REQUEST_TIMEOUT,
    retries=RETRIES,
    backoff_base_seconds=BACKOFF_BASE_SECONDS,
    use_cdn_fallback=USE_CDN_FALLBACK,
    proxy=PROXY,
)

pbp_df = normalize_api_pbpv3_df(raw_df)

print(
    f"Fetched {len(raw_df):,} raw rows and {len(pbp_df):,} normalized rows "
    f"for game {normalize_game_id(GAME_ID)} via {source_used}"
)
pbp_df.head(8)


Fetched 438 raw rows and 438 normalized rows for game 0021201230 via nba_api_stats


,actionNumber,clock,period,teamId,teamTricode,personId,playerName,playerNameI,xLegacy,yLegacy,shotDistance,shotResult,isFieldGoal,scoreHome,scoreAway,pointsTotal,location,description,actionType,subType,videoAvailable,shotValue,actionId,gameId
0,0,PT12M00.00S,1,0,,0,,,0,0,0,,0,0,0,0,,Start of 1st Period (10:45 PM EST),period,start,0,0,1,0021201230
1,1,PT12M00.00S,1,1610612757,POR,201581,Hickson,J. Hickson,0,0,0,,0,0,0,0,h,Jump Ball Hickson vs. Bogut: Tip to Curry,Jump Ball,,0,0,2,0021201230
2,2,PT11M43.00S,1,1610612744,GSW,202691,Thompson,K. Thompson,155,153,22,Missed,1,0,0,0,v,MISS Thompson 22' Jump Shot,Missed Shot,Jump Shot,0,2,3,0021201230
3,3,PT11M42.00S,1,1610612757,POR,201581,Hickson,J. Hickson,0,0,0,,0,0,0,0,h,Hickson REBOUND (Off:0 Def:1),Rebound,Unknown,0,0,4,0021201230
4,4,PT11M32.00S,1,1610612757,POR,200746,Aldridge,L. Aldridge,-11,208,21,Missed,1,0,0,0,h,MISS Aldridge 21' Jump Shot,Missed Shot,Jump Shot,0,2,5,0021201230
5,5,PT11M31.00S,1,1610612744,GSW,101106,Bogut,A. Bogut,0,0,0,,0,0,0,0,v,Bogut REBOUND (Off:0 Def:1),Rebound,Unknown,0,0,6,0021201230
6,6,PT11M25.00S,1,1610612744,GSW,201939,Curry,S. Curry,111,242,27,Missed,1,0,0,0,v,MISS Curry 27' 3PT Jump Shot,Missed Shot,Jump Shot,0,3,7,0021201230
7,7,PT11M23.00S,1,1610612757,POR,200746,Aldridge,L. Aldridge,0,0,0,,0,0,0,0,h,Aldridge REBOUND (Off:0 Def:1),Rebound,Unknown,0,0,8,0021201230


## Step 4 - Quick validation checks


In [131]:
summary = {
    "game_id": pbp_df["gameId"].iloc[0],
    "row_count": int(len(pbp_df)),
    "periods": sorted(pbp_df["period"].dropna().astype(int).unique().tolist()),
    "duplicate_action_numbers": int(pbp_df["actionNumber"].duplicated().sum()),
    "max_score_home": None if pbp_df["scoreHome"].dropna().empty else int(pbp_df["scoreHome"].dropna().max()),
    "max_score_away": None if pbp_df["scoreAway"].dropna().empty else int(pbp_df["scoreAway"].dropna().max()),
}
summary


{'game_id': '0021201230',
 'row_count': 438,
 'periods': [1, 2, 3, 4],
 'duplicate_action_numbers': 17,
 'max_score_home': 88,
 'max_score_away': 99}

In [132]:
pbp_df[["period", "clock", "actionNumber", "actionType", "subType", "description", "scoreHome", "scoreAway"]].tail(15)


,period,clock,actionNumber,actionType,subType,description,scoreHome,scoreAway
423,4,PT01M11.00S,461,Foul,Personal,Bazemore P.FOUL (P1.T3),0,0
424,4,PT01M11.00S,462,Substitution,,SUB: Barnes FOR Ezeli,0,0
425,4,PT01M00.00S,463,Foul,Shooting,Jefferson S.FOUL (P1.PN),0,0
426,4,PT01M00.00S,464,Free Throw,Free Throw 1 of 2,MISS Freeland Free Throw 1 of 2,0,0
427,4,PT01M00.00S,465,Rebound,Normal Rebound,TRAIL BLAZERS Rebound,0,0
428,4,PT01M00.00S,466,Free Throw,Free Throw 2 of 2,Freeland Free Throw 2 of 2 (7 PTS),86,97
429,4,PT00M46.90S,468,Missed Shot,Jump Shot,MISS Bazemore 10' Jump Shot,0,0
430,4,PT00M45.90S,469,Rebound,Unknown,Claver REBOUND (Off:0 Def:3),0,0
431,4,PT00M35.30S,470,Made Shot,Layup Shot,Barton 2' Layup (15 PTS),88,97
432,4,PT00M28.50S,472,Foul,Personal,Lillard P.FOUL (P2.PN),0,0


## Step 5 - Save one-off files (recommended first)

This writes game-level files you can inspect before touching season-level CSVs.


In [133]:
gid = normalize_game_id(GAME_ID)

if SAVE_ONE_OFF:
    ONE_OFF_DIR.mkdir(parents=True, exist_ok=True)
    csv_path = ONE_OFF_DIR / f"api_pbpv3_{gid}.csv"
    json_path = ONE_OFF_DIR / f"api_pbpv3_{gid}.json"

    pbp_df.to_csv(csv_path, index=False)
    pbp_df.to_json(json_path, orient="records")

    print(f"Wrote CSV:  {csv_path}")
    print(f"Wrote JSON: {json_path}")
else:
    print("SAVE_ONE_OFF is False. Skipping one-off export.")


Wrote CSV:  /Users/robschoen/Dropbox/CC/GLA/data/pbp/raw_backfill/api_pbpv3_0021201230.csv
Wrote JSON: /Users/robschoen/Dropbox/CC/GLA/data/pbp/raw_backfill/api_pbpv3_0021201230.json


## Step 6 - Optional: append/replace this game in the season CSV

Safety behavior:
- If the game already exists in the season file, its rows are removed first.
- Then the newly fetched rows are appended.
- File is re-normalized and sorted.


In [134]:
target_path = season_target_path(SEASON, PHASE, NBA_DATA_REPO)
target_path


PosixPath('/Users/robschoen/Dropbox/CC/NBA_Data/PBPdata/api_pbpv3/regular/api_pbpv3_2012.csv')

In [135]:
if APPEND_TO_SEASON_FILE:
    gid = normalize_game_id(GAME_ID)

    if target_path.exists():
        existing_df = pd.read_csv(target_path)
        existing_df = normalize_api_pbpv3_df(existing_df)
        existing_df = existing_df[existing_df["gameId"].map(normalize_game_id) != gid]
    else:
        existing_df = pd.DataFrame(columns=PBPV3_CANONICAL_COLUMNS)

    combined_df = pd.concat([existing_df, pbp_df], ignore_index=True)
    combined_df = normalize_api_pbpv3_df(combined_df)

    target_path.parent.mkdir(parents=True, exist_ok=True)
    combined_df.to_csv(target_path, index=False)

    print(f"Updated season file: {target_path}")
    print(f"Rows written: {len(combined_df):,}")
    print(f"Distinct games: {combined_df['gameId'].nunique():,}")
else:
    print("APPEND_TO_SEASON_FILE is False. No season CSV was modified.")


APPEND_TO_SEASON_FILE is False. No season CSV was modified.
